<a href="https://colab.research.google.com/github/oscar0830/AIAgentClass/blob/dev/Group_InventoryReplenishmentAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Inventory Replenishment Agent

This notebook is adapted from the starter code to match the assignment requirements:
- Load `sales.csv`, `inventory.csv`, and `params.csv`
- Forecast daily demand per SKU using **EWMA**
- Compute **safety stock** from forecast error and `service_level`
- Simulate day-by-day inventory with **lead time**
- Decide whether to **order** or **wait**
- Respect `min_order_qty`
- Compare the agent against a simple baseline
- Print a clear **daily agent log**


In [1]:

import math
import numpy as np
import pandas as pd
from scipy.stats import norm

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## 1) Load the required CSV files

In [3]:

sales = pd.read_csv("sales.csv")
inventory = pd.read_csv("inventory.csv")
params = pd.read_csv("params.csv")

sales["date"] = pd.to_datetime(sales["date"])

print("sales.csv")
display(sales.head())
print("inventory.csv")
display(inventory.head())
print("params.csv")
display(params.head())


sales.csv


,date,sku,qty_sold
0,2026-01-01,SKU_A,20
1,2026-01-02,SKU_A,11
2,2026-01-03,SKU_A,16
3,2026-01-04,SKU_A,17
4,2026-01-05,SKU_A,18


inventory.csv


,sku,opening_stock
0,SKU_A,269
1,SKU_B,181
2,SKU_C,148


params.csv


,sku,unit_cost,holding_cost_per_day,stockout_cost,lead_time_days,min_order_qty,service_level
0,SKU_A,22.0,0.18,12.0,4,40,0.95
1,SKU_B,35.0,0.24,18.0,6,30,0.97
2,SKU_C,48.0,0.32,25.0,8,20,0.98


## 2) Clean and aggregate sales per day per SKU

In [5]:

# Aggregate sales by day and SKU
daily_sales = (
    sales.groupby(["date", "sku"], as_index=False)["qty_sold"]
    .sum()
    .sort_values(["sku", "date"])
)

# Build a complete daily calendar for each SKU so missing days become zero sales
all_dates = pd.date_range(daily_sales["date"].min(), daily_sales["date"].max(), freq="D")
all_skus = sorted(params["sku"].unique())

full_index = pd.MultiIndex.from_product([all_dates, all_skus], names=["date", "sku"])
daily_sales = (
    daily_sales.set_index(["date", "sku"])
    .reindex(full_index, fill_value=0)
    .reset_index()
    .sort_values(["sku", "date"])
)

display(daily_sales.head(10))
print("Date range:", daily_sales["date"].min().date(), "to", daily_sales["date"].max().date())
print("SKUs:", all_skus)


,date,sku,qty_sold
0,2026-01-01,SKU_A,20
3,2026-01-02,SKU_A,11
6,2026-01-03,SKU_A,16
9,2026-01-04,SKU_A,17
12,2026-01-05,SKU_A,18
15,2026-01-06,SKU_A,22
18,2026-01-07,SKU_A,22
21,2026-01-08,SKU_A,16
24,2026-01-09,SKU_A,15
27,2026-01-10,SKU_A,16


Date range: 2026-01-01 to 2026-03-31
SKUs: ['SKU_A', 'SKU_B', 'SKU_C']


## 3) Forecast demand per SKU using EWMA

In [6]:

ALPHA = 0.35  # adjustable EWMA smoothing factor

def add_ewma_forecast(group, alpha=ALPHA):
    group = group.sort_values("date").copy()
    actual = group["qty_sold"].astype(float).tolist()

    forecasts = []
    errors = []
    ewma = actual[0]  # initialize from first observation

    for i, demand in enumerate(actual):
        # Forecast for current day uses information up to prior day
        if i == 0:
            f = actual[0]
        else:
            f = ewma

        forecasts.append(float(max(0, f)))
        errors.append(float(demand - f))

        # Update EWMA with actual demand
        ewma = alpha * demand + (1 - alpha) * f

    group["forecast"] = forecasts
    group["forecast_error"] = errors
    return group

forecast_df = (
    daily_sales.groupby("sku", group_keys=False)
    .apply(add_ewma_forecast, alpha=ALPHA)
    .reset_index(drop=True)
)

display(forecast_df.head(12))


/tmp/ipykernel_4821/4049917017.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_ewma_forecast, alpha=ALPHA)


,date,sku,qty_sold,forecast,forecast_error
0,2026-01-01,SKU_A,20,20.000000,0.000000
1,2026-01-02,SKU_A,11,20.000000,-9.000000
2,2026-01-03,SKU_A,16,16.850000,-0.850000
3,2026-01-04,SKU_A,17,16.552500,0.447500
4,2026-01-05,SKU_A,18,16.709125,1.290875
5,2026-01-06,SKU_A,22,17.160931,4.839069
6,2026-01-07,SKU_A,22,18.854605,3.145395
7,2026-01-08,SKU_A,16,19.955493,-3.955493
8,2026-01-09,SKU_A,15,18.571071,-3.571071
9,2026-01-10,SKU_A,16,17.321196,-1.321196


## 4) Merge inventory and parameter inputs

In [7]:

sku_setup = (
    params.merge(inventory, on="sku", how="left")
    .sort_values("sku")
    .reset_index(drop=True)
)

display(sku_setup)


,sku,unit_cost,holding_cost_per_day,stockout_cost,lead_time_days,min_order_qty,service_level,opening_stock
0,SKU_A,22.0,0.18,12.0,4,40,0.95,269
1,SKU_B,35.0,0.24,18.0,6,30,0.97,181
2,SKU_C,48.0,0.32,25.0,8,20,0.98,148


## 5) Helper functions

In [8]:

def z_from_service_level(service_level):
    # Keep values in a safe range
    service_level = min(max(float(service_level), 0.50), 0.999)
    return norm.ppf(service_level)

def safety_stock_from_error(sigma, lead_time_days, service_level):
    z = z_from_service_level(service_level)
    return max(0.0, z * sigma * math.sqrt(max(1, int(lead_time_days))))

def summarize_metrics(log_df):
    total_demand = log_df["demand"].sum()
    total_fulfilled = log_df["fulfilled_units"].sum()
    total_unfilled = log_df["unfilled_units"].sum()

    holding_cost = log_df["holding_cost"].sum()
    stockout_cost = log_df["stockout_cost"].sum()
    order_cost = log_df["order_cost"].sum()
    total_cost = holding_cost + stockout_cost + order_cost

    fill_rate = total_fulfilled / total_demand if total_demand > 0 else 1.0

    return {
        "stockouts": int((log_df["unfilled_units"] > 0).sum()),
        "units_unfilled": float(total_unfilled),
        "fill_rate": float(fill_rate),
        "holding_cost": float(holding_cost),
        "stockout_cost": float(stockout_cost),
        "order_cost": float(order_cost),
        "total_cost": float(total_cost),
    }


## 6) Inventory simulation

In [9]:

def run_policy_simulation(forecast_df, sku_setup, policy_name="agent"):
    logs = []

    for _, row in sku_setup.iterrows():
        sku = row["sku"]
        unit_cost = float(row["unit_cost"])
        holding_cost_per_day = float(row["holding_cost_per_day"])
        stockout_cost = float(row["stockout_cost"])
        lead_time_days = int(row["lead_time_days"])
        min_order_qty = int(row["min_order_qty"])
        service_level = float(row["service_level"])
        opening_stock = float(row["opening_stock"])

        sku_data = forecast_df[forecast_df["sku"] == sku].sort_values("date").reset_index(drop=True)

        # Use historical forecast error to estimate uncertainty
        sigma = max(1.0, sku_data["forecast_error"].std(ddof=1) if len(sku_data) > 1 else 1.0)

        on_hand = opening_stock
        backlog = 0.0
        pipeline = []  # list of dicts: {"arrival_date": ..., "qty": ...}

        # Baseline parameters
        avg_daily_demand = float(max(1.0, sku_data["qty_sold"].mean()))
        baseline_reorder_point = avg_daily_demand * lead_time_days
        baseline_order_qty = max(min_order_qty, int(round(avg_daily_demand * lead_time_days * 1.5)))

        for i, day in sku_data.iterrows():
            current_date = day["date"]
            demand = float(day["qty_sold"])
            forecast = float(max(0.0, day["forecast"]))

            # Receive orders arriving today
            arriving_today = sum(p["qty"] for p in pipeline if p["arrival_date"] == current_date)
            on_hand += arriving_today
            pipeline = [p for p in pipeline if p["arrival_date"] != current_date]

            pipeline_qty = sum(p["qty"] for p in pipeline)

            # Agent policy
            if policy_name == "agent":
                future_forecasts = sku_data.iloc[i : i + lead_time_days]["forecast"]
                forecast_during_lead_time = float(future_forecasts.sum()) if len(future_forecasts) > 0 else forecast * lead_time_days
                safety_stock = safety_stock_from_error(sigma, lead_time_days, service_level)
                reorder_point = forecast_during_lead_time + safety_stock
                inventory_position = on_hand + pipeline_qty - backlog

                if inventory_position < reorder_point:
                    raw_qty = reorder_point - inventory_position
                    order_qty = max(min_order_qty, int(math.ceil(raw_qty)))
                    action = "order"
                    rationale = (
                        f"Ordered because inventory position ({inventory_position:.1f}) "
                        f"was below reorder point ({reorder_point:.1f})."
                    )
                else:
                    order_qty = 0
                    action = "wait"
                    rationale = (
                        f"Waited because inventory position ({inventory_position:.1f}) "
                        f"was above reorder point ({reorder_point:.1f}), reducing holding cost."
                    )

            # Simple baseline: fixed reorder point / fixed order quantity
            else:
                forecast_during_lead_time = avg_daily_demand * lead_time_days
                safety_stock = 0.0
                reorder_point = baseline_reorder_point
                inventory_position = on_hand + pipeline_qty - backlog

                if inventory_position < reorder_point:
                    order_qty = baseline_order_qty
                    action = "order"
                    rationale = (
                        f"Baseline ordered fixed quantity because inventory position "
                        f"({inventory_position:.1f}) was below fixed reorder point ({reorder_point:.1f})."
                    )
                else:
                    order_qty = 0
                    action = "wait"
                    rationale = "Baseline waited because inventory position remained above its fixed reorder point."

            # Place order to arrive after lead time
            order_cost = 0.0
            if order_qty > 0:
                arrival_date = current_date + pd.Timedelta(days=lead_time_days)
                pipeline.append({"arrival_date": arrival_date, "qty": order_qty})
                order_cost = 0.0  # assignment says fixed order cost can be 0 for now

            # Fulfill today's demand including backlog
            total_required = demand + backlog
            fulfilled_units = min(on_hand, total_required)
            on_hand -= fulfilled_units
            backlog = total_required - fulfilled_units
            unfilled_units = backlog

            holding_cost = on_hand * holding_cost_per_day
            stockout_cost_today = unfilled_units * stockout_cost

            logs.append({
                "policy": policy_name,
                "date": current_date,
                "sku": sku,
                "demand": demand,
                "forecast": forecast,
                "forecast_during_lead_time": forecast_during_lead_time,
                "sigma_error": sigma,
                "service_level": service_level,
                "safety_stock": safety_stock,
                "reorder_point": reorder_point,
                "inventory_position_before_decision": inventory_position,
                "action": action,
                "order_qty": order_qty,
                "arriving_today": arriving_today,
                "pipeline_qty_after_order": sum(p["qty"] for p in pipeline),
                "on_hand_end_of_day": on_hand,
                "backlog_end_of_day": backlog,
                "fulfilled_units": fulfilled_units,
                "unfilled_units": unfilled_units,
                "holding_cost": holding_cost,
                "stockout_cost": stockout_cost_today,
                "order_cost": order_cost,
                "rationale": rationale,
            })

    log_df = pd.DataFrame(logs).sort_values(["date", "sku"]).reset_index(drop=True)
    metrics = summarize_metrics(log_df)
    return log_df, metrics


## 7) Run baseline and agent simulations

In [10]:

baseline_log, baseline_metrics = run_policy_simulation(forecast_df, sku_setup, policy_name="baseline")
agent_log, agent_metrics = run_policy_simulation(forecast_df, sku_setup, policy_name="agent")

baseline_metrics, agent_metrics


({'stockouts': 33,
  'units_unfilled': 309.0,
  'fill_rate': 1.0,
  'holding_cost': 3211.98,
  'stockout_cost': 4924.0,
  'order_cost': 0.0,
  'total_cost': 8135.98},
 {'stockouts': 13,
  'units_unfilled': 124.0,
  'fill_rate': 1.0,
  'holding_cost': 1944.62,
  'stockout_cost': 1501.0,
  'order_cost': 0.0,
  'total_cost': 3445.62})

## 8) KPI comparison

In [11]:

comparison = pd.DataFrame([
    {"policy": "baseline", **baseline_metrics},
    {"policy": "agent", **agent_metrics},
])

comparison


,policy,stockouts,units_unfilled,fill_rate,holding_cost,stockout_cost,order_cost,total_cost
0,baseline,33,309.0,1.0,3211.98,4924.0,0.0,8135.98
1,agent,13,124.0,1.0,1944.62,1501.0,0.0,3445.62


## 9) Daily agent log

In [12]:

agent_log.head(25)


,policy,date,sku,demand,forecast,forecast_during_lead_time,sigma_error,service_level,safety_stock,reorder_point,inventory_position_before_decision,action,order_qty,arriving_today,pipeline_qty_after_order,on_hand_end_of_day,backlog_end_of_day,fulfilled_units,unfilled_units,holding_cost,stockout_cost,order_cost,rationale
0,agent,2026-01-01,SKU_A,20.0,20.000000,73.402500,4.061956,0.95,13.362647,86.765147,269.0,wait,0,0,0,249.0,0.0,20.0,0.0,44.82,0.0,0.0,Waited because inventory position (269.0) was ...
1,agent,2026-01-01,SKU_B,12.0,12.000000,71.169625,3.326061,0.97,15.323111,86.492736,181.0,wait,0,0,0,169.0,0.0,12.0,0.0,40.56,0.0,0.0,Waited because inventory position (181.0) was ...
2,agent,2026-01-01,SKU_C,8.0,8.000000,62.471041,2.051368,0.98,11.916151,74.387192,148.0,wait,0,0,0,140.0,0.0,8.0,0.0,44.80,0.0,0.0,Waited because inventory position (148.0) was ...
3,agent,2026-01-02,SKU_A,11.0,20.000000,70.111625,4.061956,0.95,13.362647,83.474272,249.0,wait,0,0,0,238.0,0.0,11.0,0.0,42.84,0.0,0.0,Waited because inventory position (249.0) was ...
4,agent,2026-01-02,SKU_B,12.0,12.000000,72.510256,3.326061,0.97,15.323111,87.833367,169.0,wait,0,0,0,157.0,0.0,12.0,0.0,37.68,0.0,0.0,Waited because inventory position (169.0) was ...
5,agent,2026-01-02,SKU_C,10.0,8.000000,62.656177,2.051368,0.98,11.916151,74.572327,140.0,wait,0,0,0,130.0,0.0,10.0,0.0,41.60,0.0,0.0,Waited because inventory position (140.0) was ...
6,agent,2026-01-03,SKU_A,16.0,16.850000,67.272556,4.061956,0.95,13.362647,80.635203,238.0,wait,0,0,0,222.0,0.0,16.0,0.0,39.96,0.0,0.0,Waited because inventory position (238.0) was ...
7,agent,2026-01-03,SKU_B,11.0,12.000000,75.131667,3.326061,0.97,15.323111,90.454777,157.0,wait,0,0,0,146.0,0.0,11.0,0.0,35.04,0.0,0.0,Waited because inventory position (157.0) was ...
8,agent,2026-01-03,SKU_C,4.0,8.700000,61.726515,2.051368,0.98,11.916151,73.642665,130.0,wait,0,0,0,126.0,0.0,4.0,0.0,40.32,0.0,0.0,Waited because inventory position (130.0) was ...
9,agent,2026-01-04,SKU_A,17.0,16.552500,69.277162,4.061956,0.95,13.362647,82.639808,222.0,wait,0,0,0,205.0,0.0,17.0,0.0,36.90,0.0,0.0,Waited because inventory position (222.0) was ...


## 10) Trade-off reflection

In [13]:

tradeoff_reflection = '''
The agent tries to balance service reliability against inventory cost. By forecasting demand and adding safety stock based on forecast error and service level, it is more likely to avoid stockouts than a simple fixed reorder rule. That usually improves fill rate and customer trust, but it can also raise holding cost if the service target is high.

The key business trade-off is that carrying more inventory protects service levels, while carrying less inventory reduces working capital and storage cost. Daily logs improve trust because managers can see why the agent ordered or waited on each SKU.
'''
print(tradeoff_reflection)



The agent tries to balance service reliability against inventory cost. By forecasting demand and adding safety stock based on forecast error and service level, it is more likely to avoid stockouts than a simple fixed reorder rule. That usually improves fill rate and customer trust, but it can also raise holding cost if the service target is high.

The key business trade-off is that carrying more inventory protects service levels, while carrying less inventory reduces working capital and storage cost. Daily logs improve trust because managers can see why the agent ordered or waited on each SKU.



## 11) Scaling note

In [14]:

scaling_note = '''
Cloud deployment plan:
This agent could run as a scheduled daily job in a container on AWS, Azure, or GCP. Each run would load fresh sales and inventory data, recompute forecasts, and write recommended purchase orders to a database or dashboard.

Monitoring:
Track fill rate, stockouts, forecast error, and inventory position by SKU. Add alerts when fill rate drops below target, demand spikes, or backlog persists for multiple days.

Reliability:
Add input validation for missing CSV fields, retry logic for failed data loads, and fallback rules such as a naive forecast when demand history is too short or corrupted.

Cost controls:
Add max inventory caps, working-capital limits, and approval thresholds for large orders so the agent does not over-order even when demand uncertainty increases.
'''
print(scaling_note)



Cloud deployment plan:
This agent could run as a scheduled daily job in a container on AWS, Azure, or GCP. Each run would load fresh sales and inventory data, recompute forecasts, and write recommended purchase orders to a database or dashboard.

Monitoring:
Track fill rate, stockouts, forecast error, and inventory position by SKU. Add alerts when fill rate drops below target, demand spikes, or backlog persists for multiple days.

Reliability:
Add input validation for missing CSV fields, retry logic for failed data loads, and fallback rules such as a naive forecast when demand history is too short or corrupted.

Cost controls:
Add max inventory caps, working-capital limits, and approval thresholds for large orders so the agent does not over-order even when demand uncertainty increases.

